# Movement notebook

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('/root/movement')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# %% [markdown]
# Notebook v2 — Humanoid-X style dataset builder (2D -> 3D -> retargeting)
# - 2D: ViTPose++ (HF Transformers)
# - 3D: MotionAGFormer (2D->3D lifting)
# - Retargeting: PyBullet IK (no PoseLib)
# - Optional: per-frame screenshots with ffmpeg

# %% [code]
import os, json, math, glob, shutil, subprocess, warnings

import numpy as np
import cv2
import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using CUDA: {gpu_name}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
from transformers import AutoProcessor, VitPoseForPoseEstimation
import tqdm as notebook_tqdm

MODEL_REPO = "usyd-community/vitpose-plus-large"
image_processor = AutoProcessor.from_pretrained(MODEL_REPO)
model = VitPoseForPoseEstimation.from_pretrained(MODEL_REPO)
model.to(device)
model.eval()
print("VitPose loaded on", device)


In [ ]:
# Parameters (use local Kinetics-700-2020 samples; deterministic hardcoded)
DATA_DIR = Path("/root/movement/data/kinetics-dataset/k700-2020/train")
SAMPLES = [
    "acting in play/0bdVrgImymc_000020_000030",
    "acting in play/0bk_zXtqOTQ_000020_000030",
    "acting in play/0lZhBK_QOak_000283_000293",
]
# Choose which sample to run below
filename = SAMPLES[2]
person_id = "1"
print("Data dir:", DATA_DIR.resolve())
print("Selected:", filename)

In [ ]:
# Local-only check for Kinetics raw file
raw_candidate = (DATA_DIR / f"{filename}.mp4").resolve()
raw_video_path = str(raw_candidate)
print("Raw candidate:", raw_candidate, "exists:", raw_candidate.exists())

In [ ]:
# Generate tracking JSON with simple IoU-based tracking (person only)
import json
import torch
import torchvision
import torchvision.transforms as T
import cv2
import numpy as np

tracking_path = (DATA_DIR / f"{filename}.json").resolve()
print("Writing tracking to:", tracking_path)

# Load detector
model_det = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT").to(device).eval()
transform = T.ToTensor()

# IoU helper
from data_prep.boxes import iou_xyxy

# Iterate frames and track single identity via IoU continuity
frame_idxs = []
boxes_xyxy = []
last_box = None
iou_thresh = 0.3
score_thresh = 0.7
cap = cv2.VideoCapture(raw_video_path)
i = 0
with torch.no_grad():
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        img = transform(frame_rgb).to(device)
        out = model_det([img])[0]
        chosen = None
        if out["boxes"].numel() > 0:
            labels = out["labels"].detach().cpu().numpy()
            scores = out["scores"].detach().cpu().numpy()
            boxes = out["boxes"].detach().cpu().numpy()  # xyxy
            mask = (labels == 1) & (scores >= score_thresh)
            cand_idx = np.flatnonzero(mask)
            if cand_idx.size > 0:
                if last_box is not None:
                    # pick by max IoU, fallback to best score
                    ious = [iou_xyxy(last_box, boxes[j]) for j in cand_idx]
                    j_rel = int(np.argmax(ious))
                    if ious[j_rel] >= iou_thresh:
                        chosen = boxes[cand_idx[j_rel]]
                if chosen is None:
                    # fallback to highest score candidate
                    j_rel = int(np.argmax(scores[cand_idx]))
                    chosen = boxes[cand_idx[j_rel]]
        if chosen is not None:
            x1, y1, x2, y2 = map(float, chosen.tolist())
            # Optional: enlarge box slightly to reduce crop cutoffs
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            w, h = (x2 - x1), (y2 - y1)
            scale = 1.15
            nw, nh = w * scale, h * scale
            x1e, y1e = max(0.0, cx - nw / 2), max(0.0, cy - nh / 2)
            x2e, y2e = cx + nw / 2, cy + nh / 2
            frame_idxs.append(i)
            boxes_xyxy.append([x1e, y1e, x2e, y2e])
            last_box = np.array([x1e, y1e, x2e, y2e], dtype=np.float32)
        i += 1
cap.release()

tracking = {str(person_id): {"frame_idxs": frame_idxs, "box": boxes_xyxy}}
tracking_path.parent.mkdir(parents=True, exist_ok=True)
with open(tracking_path, "w") as f:
    json.dump(tracking, f)
print(f"Saved {len(frame_idxs)} tracked frames for person_id={person_id}")



In [ ]:
import json
import cv2
import numpy as np
import torch
import PIL

# Load tracking JSON and use first tracked frame/box
tracking_path = (DATA_DIR / f"{filename}.json").resolve()
with open(tracking_path, "r") as f:
    tracking = json.load(f)

idxs = tracking[str(person_id)]["frame_idxs"]
boxes_xyxy = tracking[str(person_id)]["box"]
if not idxs:
    raise RuntimeError("Tracking JSON has no frames")

target_idx = int(idxs[0])
xyxy = boxes_xyxy[0]  # [x1,y1,x2,y2]
# convert to COCO [x,y,w,h] as (1,4) float32
person_boxes_coco = np.expand_dims(np.array([xyxy[0], xyxy[1], xyxy[2]-xyxy[0], xyxy[3]-xyxy[1]], dtype=np.float32), axis=0)

# Seek and read that frame
cap = cv2.VideoCapture(raw_video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"Failed to read frame {target_idx}")

h, w = frame_bgr.shape[:2]
rgb_frame = frame_bgr[:, :, ::-1]
image = PIL.Image.fromarray(rgb_frame)

# Prepare inputs for ViTPose with box (exact pattern from metabot_gendata)
pixel_values = image_processor(image, boxes=[person_boxes_coco], return_tensors="pt").pixel_values

# dataset_index as 1D tensor on device
dataset_index = torch.tensor([0], device=device)

pixel_values = pixel_values.to(device=device)
with torch.no_grad():
    outputs = model(pixel_values, dataset_index=dataset_index)

pose_results = image_processor.post_process_pose_estimation(outputs, boxes=[person_boxes_coco])
image_pose_result = pose_results[0][0]  # results for first image

xy = np.expand_dims(image_pose_result["keypoints"].cpu().numpy(), axis=0)
scores = np.expand_dims(image_pose_result["scores"].cpu().numpy(), axis=0)

print("frame:", target_idx, "num_persons:", 1)
print("xy:", xy.shape, "scores:", scores.shape)

xy, scores


In [ ]:
from PIL import Image

assert 'pixel_values' in locals(), "pixel_values not available; run the pose cell first."

batch_index = 0
mean = image_processor.image_mean
std = image_processor.image_std
img_np = pixel_values[batch_index].detach().cpu().numpy()
unnormalized = (img_np * np.array(std)[:, None, None]) + np.array(mean)[:, None, None]
unnormalized = (unnormalized * 255).astype(np.uint8)
unnormalized = np.moveaxis(unnormalized, 0, -1)
Image.fromarray(unnormalized)


In [ ]:
import matplotlib.pyplot as plt

assert 'outputs' in locals(), "outputs not available; run the pose cell first."
assert hasattr(outputs, 'heatmaps'), "This model checkpoint does not expose raw heatmaps."

from data_prep.visualization import visualize_layered_heatmaps

visualize_layered_heatmaps(outputs.heatmaps.detach().cpu().numpy())


In [ ]:
import supervision as sv
import numpy as np
import matplotlib.pyplot as plt

assert 'xy' in locals() and xy is not None, "xy not available; run the pose cell first."
assert 'scores' in locals() and scores is not None, "scores not available; run the pose cell first."
assert 'rgb_frame' in locals(), "rgb_frame not available; run the pose cell first."
assert 'xyxy' in locals(), "xyxy (bbox) not available; run the pose cell first."

annot_img = rgb_frame.copy()

# Build KeyPoints object as in metabot_gendata
keypoints = sv.KeyPoints(xy=xy.astype(np.float32), confidence=scores.astype(np.float32))
# Build Detections for optional bbox drawing (expects xyxy)
detections = sv.Detections(xyxy=np.expand_dims(np.array(xyxy, dtype=np.float32), axis=0))

edge_annotator = sv.EdgeAnnotator(color=sv.Color.GREEN, thickness=3)
vertex_annotator = sv.VertexAnnotator(color=sv.Color.RED, radius=2)
bounding_box_annotator = sv.BoxAnnotator(color=sv.Color.BLUE, color_lookup=sv.ColorLookup.INDEX, thickness=1)

# Annotate
annot_img = edge_annotator.annotate(scene=annot_img, key_points=keypoints)
annot_img = vertex_annotator.annotate(scene=annot_img, key_points=keypoints)
annot_img = bounding_box_annotator.annotate(scene=annot_img, detections=detections)

plt.figure(figsize=(6,6))
plt.imshow(annot_img)
plt.axis('off')
plt.title('ViTPose keypoints')
plt.show()


In [ ]:
import torch
import torch.nn as nn

MAGF_ROOT = Path('/root/movement/MotionAGFormer')
assert MAGF_ROOT.exists(), f"Missing MotionAGFormer repo at {MAGF_ROOT}"
# Ensure both parent and repo are on sys.path so `import MotionAGFormer` works
PARENT = MAGF_ROOT.parent
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
if str(MAGF_ROOT) not in sys.path:
    sys.path.insert(0, str(MAGF_ROOT))

from MotionAGFormer.model.MotionAGFormer import MotionAGFormer

n_layers, dim_in, dim_feat, dim_rep, dim_out = 16, 3, 128, 512, 3
mlp_ratio, act_layer = 4, nn.GELU
attn_drop, drop, drop_path = 0.0, 0.0, 0.0
use_layer_scale, layer_scale_init_value, use_adaptive_fusion = True, 1e-5, True
num_heads, qkv_bias, qkv_scale = 8, False, None
hierarchical = False
use_temporal_similarity, neighbour_num, temporal_connection_len = True, 2, 1
use_tcn, graph_only = False, False
n_frames = 243

model_3d = nn.DataParallel(
    MotionAGFormer(
        n_layers=n_layers,
        dim_in=dim_in,
        dim_feat=dim_feat,
        dim_rep=dim_rep,
        dim_out=dim_out,
        mlp_ratio=mlp_ratio,
        act_layer=act_layer,
        attn_drop=attn_drop,
        drop=drop,
        drop_path=drop_path,
        use_layer_scale=use_layer_scale,
        layer_scale_init_value=layer_scale_init_value,
        use_adaptive_fusion=use_adaptive_fusion,
        num_heads=num_heads,
        qkv_bias=qkv_bias,
        qkv_scale=qkv_scale,
        hierarchical=hierarchical,
        use_temporal_similarity=use_temporal_similarity,
        neighbour_num=neighbour_num,
        temporal_connection_len=temporal_connection_len,
        use_tcn=use_tcn,
        graph_only=graph_only,
        n_frames=n_frames,
    )
).to(device)

ckpt_path = Path('/root/movement/models/motionagformer-b-h36m.pth.tr')
assert ckpt_path.exists(), f"Checkpoint not found at {ckpt_path}"
pre_dict = torch.load(str(ckpt_path), map_location=device, weights_only=False)
assert 'model' in pre_dict, "Checkpoint missing 'model' key"
model_3d.load_state_dict(pre_dict['model'], strict=True)
model_3d.eval()
print('MotionAGFormer loaded from', ckpt_path)


In [ ]:
import numpy as np

from data_prep.temporal import resample
from data_prep.geometry import normalize_to_bbox
from data_prep.temporal import to_clips
from data_prep.keypoints import flip_magformer

def turn_into_clips(keypoints):
    clips = []
    n_frames = keypoints.shape[1]
    if n_frames <= 243:
        new_indices = resample(n_frames)
        clips.append(keypoints[:, new_indices, ...])
        downsample = np.unique(new_indices, return_index=True)[1]
    else:
        for start_idx in range(0, n_frames, 243):
            keypoints_clip = keypoints[:, start_idx:start_idx + 243, ...]
            clip_length = keypoints_clip.shape[1]
            if clip_length != 243:
                new_indices = resample(clip_length)
                clips.append(keypoints_clip[:, new_indices, ...])
                downsample = np.unique(new_indices, return_index=True)[1]
            else:
                clips.append(keypoints_clip)
    return clips, downsample

h36m_coco_order = [9, 11, 14, 12, 15, 13, 16, 4, 1, 5, 2, 6, 3]
coco_order = [0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
spple_keypoints = [10, 8, 0, 7]


def coco_h36m(keypoints):
    temporal = keypoints.shape[0]
    keypoints_h36m = np.zeros_like(keypoints, dtype=np.float32)
    htps_keypoints = np.zeros((temporal, 4, 2), dtype=np.float32)

    # htps_keypoints: head, thorax, pelvis, spine
    htps_keypoints[:, 0, 0] = np.mean(keypoints[:, 1:5, 0], axis=1, dtype=np.float32)
    htps_keypoints[:, 0, 1] = np.sum(keypoints[:, 1:3, 1], axis=1, dtype=np.float32) - keypoints[:, 0, 1]
    htps_keypoints[:, 1, :] = np.mean(keypoints[:, 5:7, :], axis=1, dtype=np.float32)
    htps_keypoints[:, 1, :] += (keypoints[:, 0, :] - htps_keypoints[:, 1, :]) / 3

    htps_keypoints[:, 2, :] = np.mean(keypoints[:, 11:13, :], axis=1, dtype=np.float32)
    htps_keypoints[:, 3, :] = np.mean(keypoints[:, [5, 6, 11, 12], :], axis=1, dtype=np.float32)

    keypoints_h36m[:, spple_keypoints, :] = htps_keypoints
    keypoints_h36m[:, h36m_coco_order, :] = keypoints[:, coco_order, :]

    keypoints_h36m[:, 9, :] -= (keypoints_h36m[:, 9, :] - np.mean(keypoints[:, 5:7, :], axis=1, dtype=np.float32)) / 4
    keypoints_h36m[:, 7, 0] += 2*(keypoints_h36m[:, 7, 0] - np.mean(keypoints_h36m[:, [0, 8], 0], axis=1, dtype=np.float32))
    keypoints_h36m[:, 8, 1] -= (np.mean(keypoints[:, 1:3, 1], axis=1, dtype=np.float32) - keypoints[:, 0, 1])*2/3

    valid_frames = np.where(np.sum(keypoints_h36m.reshape(-1, 34), axis=1) != 0)[0]
    
    return keypoints_h36m, valid_frames


def h36m_coco_format(keypoints, scores):
    assert len(keypoints.shape) == 4 and len(scores.shape) == 3

    h36m_kpts = []
    h36m_scores = []
    valid_frames = []

    for i in range(keypoints.shape[0]):
        kpts = keypoints[i]
        score = scores[i]

        new_score = np.zeros_like(score, dtype=np.float32)

        if np.sum(kpts) != 0.:
            kpts, valid_frame = coco_h36m(kpts)
            h36m_kpts.append(kpts)
            valid_frames.append(valid_frame)

            new_score[:, h36m_coco_order] = score[:, coco_order]
            new_score[:, 0] = np.mean(score[:, [11, 12]], axis=1, dtype=np.float32)
            new_score[:, 8] = np.mean(score[:, [5, 6]], axis=1, dtype=np.float32)
            new_score[:, 7] = np.mean(new_score[:, [0, 8]], axis=1, dtype=np.float32)
            new_score[:, 10] = np.mean(score[:, [1, 2, 3, 4]], axis=1, dtype=np.float32)

            h36m_scores.append(new_score)

    h36m_kpts = np.asarray(h36m_kpts, dtype=np.float32)
    h36m_scores = np.asarray(h36m_scores, dtype=np.float32)

    return h36m_kpts, h36m_scores, valid_frames


In [ ]:
from data_prep.keypoints import coco_h36m, h36m_coco_format


In [ ]:
# Extract 2D keypoints sequence across tracked frames (single person)
import json
import PIL
import torch
import numpy as np
import cv2

# Preconditions
assert 'image_processor' in globals(), 'image_processor not initialized'
assert 'model' in globals(), 'ViTPose model not initialized'

# Load tracking
tracking_path = (DATA_DIR / f"{filename}.json").resolve()
with open(tracking_path, 'r') as f:
    tracking = json.load(f)
idxs = tracking[str(person_id)]['frame_idxs']
boxes_xyxy = tracking[str(person_id)]['box']
assert len(idxs) == len(boxes_xyxy) and len(idxs) > 0, 'Empty or mismatched tracking'

cap = cv2.VideoCapture(raw_video_path)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

all_keypoints = np.zeros((1, len(idxs), 17, 2), dtype=np.float32)
all_scores = np.zeros((1, len(idxs), 17), dtype=np.float32)

current_idx_idx = 0
for i in range(num_frames):
    ok, frame_bgr = cap.read()
    if not ok:
        break
    if current_idx_idx < len(idxs) and i == int(idxs[current_idx_idx]):
        rgb_frame = frame_bgr[:, :, ::-1]
        bbox_xyxy = np.array(boxes_xyxy[current_idx_idx], dtype=np.float32)
        person_boxes_coco = np.expand_dims(np.array([bbox_xyxy[0], bbox_xyxy[1], bbox_xyxy[2]-bbox_xyxy[0], bbox_xyxy[3]-bbox_xyxy[1]], dtype=np.float32), axis=0)

        image = PIL.Image.fromarray(rgb_frame)
        pixel_values = image_processor(image, boxes=[person_boxes_coco], return_tensors='pt').pixel_values
        dataset_index = torch.tensor([0], device=device)

        pixel_values = pixel_values.to(device=device)
        with torch.no_grad():
            outputs = model(pixel_values, dataset_index=dataset_index)
        pose_results = image_processor.post_process_pose_estimation(outputs, boxes=[person_boxes_coco])
        image_pose_result = pose_results[0][0]
        all_keypoints[0, current_idx_idx] = image_pose_result['keypoints'].cpu().numpy()
        all_scores[0, current_idx_idx] = image_pose_result['scores'].cpu().numpy()

        current_idx_idx += 1
        if current_idx_idx >= len(idxs):
            break

cap.release()

print('2D sequence extracted:', all_keypoints.shape, all_scores.shape)

seq_keypoints, seq_scores, valid_frames = h36m_coco_format(all_keypoints, all_scores)
combined = np.concatenate((seq_keypoints, seq_scores[..., None]), axis=-1)
clips, downsample = turn_into_clips(combined)
print('Prepared clips:', len(clips), 'downsample idx len:', len(downsample))

In [ ]:
from data_prep.geometry import normalize_screen

# 1) Build (x,y,conf) arrays in H36M order (you already do this):
seq_xy  = seq_keypoints.astype(np.float32)              # (1,F,17,2) pixels
seq_c   = seq_scores[..., None].astype(np.float32)      # (1,F,17,1)

# 2) GLOBAL normalization (not bbox)
xy_norm = normalize_screen(seq_xy, width, height)       # (1,F,17,2)
inp_xyc = np.concatenate([xy_norm, seq_c], axis=-1)     # (1,F,17,3)

# 3) Clip/resample with a per-clip index map
clips, idx_maps = [], []
T = inp_xyc.shape[1]
if T <= 243:
    idx = np.floor(np.linspace(0, T-1, 243)).astype(np.int32)
    clips.append(inp_xyc[:, idx])
    idx_maps.append(np.unique(idx, return_index=True)[1])
else:
    for s in range(0, T, 243):
        chunk = inp_xyc[:, s:s+243]
        if chunk.shape[1] < 243:
            idx = np.floor(np.linspace(0, chunk.shape[1]-1, 243)).astype(np.int32)
            clips.append(chunk[:, idx])
            idx_maps.append(np.unique(idx, return_index=True)[1])
        else:
            clips.append(chunk)
            idx_maps.append(None)

# 4) Inference (keep flipped ensemble; keep confidence channel)
from data_prep.keypoints import flip_magformer

all_3d = []
model_3d.eval()
with torch.no_grad():
    for ci, clip in enumerate(clips):
        t_in = torch.from_numpy(clip).to(device)          # (1,243,17,3)

        out_nf = model_3d(t_in)
        out_f  = model_3d(flip_magformer(t_in))
        out    = 0.5 * (out_nf + flip_magformer(out_f))

        # root-center by subtraction (safer than just zeroing)
        out = out - out[:, :, 0:1, :]

        if idx_maps[ci] is not None:
            out = out[:, idx_maps[ci]]

        all_3d.append(out[0].cpu().numpy())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# H36M edges (0-indexed)
I = np.array([0,1,2,0,4,5,0,7,8,9,8,11,12,8,14,15])
J = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16])

from data_prep.visualization import plot_pose_equal_axes

# usage on your first frame of the first clip:
plot_pose_equal_axes(all_3d[0][0], title="3D Pose (clip 0, frame 0)")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ----- config -----
# which sequence index to draw (0 = first tracked frame)
seq_idx = 0
conf_thresh = 0.15          # hide very low-conf joints
radius      = 3
thickness   = 2

# H36M edges (0-indexed)
H36M_I = np.array([0,1,2,0,4,5,0,7,8,9,8,11,12,8,14,15])
H36M_J = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16])

# Optional: left/right coloring (bool mask per edge)
LR = np.array([0,0,0,1,1,1,0,0,0,0,1,1,1,0,0,0], dtype=bool)
COLOR_L = (0,255,0)  # BGR for "left"
COLOR_R = (255,0,0)  # BGR for "right"

# ----- grab the RGB frame that corresponds to this seq index -----
assert 'raw_video_path' in globals() and 'idxs' in globals()
frame_no = int(idxs[seq_idx])

cap = cv2.VideoCapture(raw_video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
ok, frame_bgr = cap.read()
cap.release()
assert ok, f"Failed to read frame {frame_no}"

# ----- 2D H36M points & conf in PIXELS (not normalized) -----
pts    = seq_keypoints[0, seq_idx].copy()   # (17,2) pixels
scores = seq_scores[0,    seq_idx].copy()   # (17,)

# safety: clip to image bounds before drawing
h, w = frame_bgr.shape[:2]
pts[:,0] = np.clip(pts[:,0], 0, w-1)
pts[:,1] = np.clip(pts[:,1], 0, h-1)

# ----- draw skeleton on a copy -----
canvas = frame_bgr.copy()

# draw edges
for i, j, is_left in zip(H36M_I, H36M_J, LR):
    if scores[i] < conf_thresh or scores[j] < conf_thresh: 
        continue
    p1 = tuple(np.round(pts[i]).astype(int))
    p2 = tuple(np.round(pts[j]).astype(int))
    color = COLOR_L if is_left else COLOR_R
    cv2.line(canvas, p1, p2, color, thickness, cv2.LINE_AA)

# draw joints
for k in range(pts.shape[0]):
    if scores[k] < conf_thresh:
        continue
    p = tuple(np.round(pts[k]).astype(int))
    cv2.circle(canvas, p, radius, (0,255,255), -1, cv2.LINE_AA)  # yellow dots

# ----- show -----
canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(7,6))
plt.imshow(canvas_rgb)
plt.title(f'H36M overlay — raw frame {frame_no} (seq idx {seq_idx})')
plt.axis('off')
plt.show()


In [ ]:
import os
import cv2
import numpy as np
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from matplotlib import animation

# Inline 3D overlay (no files written): compressed embed with subsampling and downscaling
assert 'all_3d' in globals() and len(all_3d) > 0, "Run the 3D lifting cell to create all_3d."
assert 'seq_keypoints' in globals() and 'idxs' in globals() and 'raw_video_path' in globals(), "Missing prerequisites (seq_keypoints, idxs, raw_video_path)."

# Tunables to stay under embed size limit
stride = 2           # keep every Nth frame
resize_scale = 0.5   # downscale frames (0.5 = half-size)
max_fps = 20.0       # cap output fps

# Concatenate clips and align lengths with 2D sequence
y3d = np.concatenate(all_3d, axis=0)  # (T3, 17, 3)
T = min(y3d.shape[0], len(idxs), seq_keypoints.shape[1])
if T < len(idxs):
    print(f"Note: 3D predictions length ({y3d.shape[0]}) < tracked frames ({len(idxs)}); truncating to {T}.")

# Fit 2D similarity per-frame to map 3D XY -> pixel space of 2D detections
H36M_I = np.array([0,1,2,0,4,5,0,7,8,9,8,11,12,8,14,15])
H36M_J = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16])

from data_prep.geometry import fit_similarity_2d

# Probe source video for fps and size (for timing/figsize only)
cap0 = cv2.VideoCapture(raw_video_path)
fps_src = cap0.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap0.release()

# Build frames in memory (subsample + resize)
frames_rgb = []
for s in range(0, T, int(max(1, stride))):
    frame_no = int(idxs[s])
    cap = cv2.VideoCapture(raw_video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        continue

    P = y3d[s]
    X = P[:, :2].astype(np.float32)
    Z = P[:, 2].astype(np.float32)
    Y = seq_keypoints[0, s].astype(np.float32)

    scale_xy, R, t = fit_similarity_2d(X, Y)
    mapped = (X @ R) * scale_xy + t  # (17,2) pixels

    z_edge = ((Z[H36M_I] + Z[H36M_J]) * 0.5)
    order = np.argsort(z_edge)

    canvas = frame_bgr.copy()
    for idx_e in order:
        i = int(H36M_I[idx_e]); j = int(H36M_J[idx_e])
        p1 = tuple(np.round(mapped[i]).astype(int))
        p2 = tuple(np.round(mapped[j]).astype(int))
        cv2.line(canvas, p1, p2, (0, 180, 255), 2, cv2.LINE_AA)
    for k in range(mapped.shape[0]):
        p = tuple(np.round(mapped[k]).astype(int))
        cv2.circle(canvas, p, 3, (255, 255, 0), -1, cv2.LINE_AA)

    if resize_scale != 1.0:
        canvas = cv2.resize(
            canvas,
            (int(canvas.shape[1] * resize_scale), int(canvas.shape[0] * resize_scale)),
            interpolation=cv2.INTER_AREA,
        )

    frames_rgb.append(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))

# Create and display animation inline (HTML5 video for better compression)
fps_out = float(min(max_fps, fps_src / max(1.0, float(stride))))
fig, ax = plt.subplots(figsize=(frames_rgb[0].shape[1]/160, frames_rgb[0].shape[0]/160), dpi=160)
ax.axis('off')
im = ax.imshow(frames_rgb[0])

def update(i):
    im.set_data(frames_rgb[i])
    return [im]

anim = animation.FuncAnimation(fig, update, frames=len(frames_rgb), interval=1000.0/float(fps_out), blit=True)
html = anim.to_html5_video()
display(HTML(html))
plt.close(fig)
print(f"Rendered inline 3D overlay (no files). frames={len(frames_rgb)} stride={stride} scale={resize_scale} fps~{fps_out:.1f}")

In [ ]:
from data_prep.geometry import fit_similarity_2d
from data_prep.visualization import overlay_3d_on_video_inline

stride = 2
resize_scale = 0.5
max_fps = 20.0

overlay_3d_on_video_inline(
    all_3d=all_3d,
    seq_keypoints=seq_keypoints,
    idxs=idxs,
    raw_video_path=raw_video_path,
    fit_similarity_2d_fn=fit_similarity_2d,
    stride=stride,
    resize_scale=resize_scale,
    max_fps=max_fps,
)